# B2-019-attention-transformers — Practice p10 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** multi-head-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

Write d = head*d_h + coordinate. Splitting maps x[b,n,d] to heads[b,head,n,coordinate] by reshaping (B,N,D) to (B,N,h,d_h) and transposing axes 1 and 2. Concatenation applies the inverse transpose before restoring the last two axes to width D.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 0.0
RTOL = 0.0

def split_heads(x, h):
    if not isinstance(x, np.ndarray) or x.ndim != 3:
        raise ValueError("x must be a rank-three NumPy array")
    b, n, d = x.shape
    if h <= 0 or d % h != 0:
        raise ValueError("h must be positive and divide D")
    return x.reshape(b, n, h, d // h).transpose(0, 2, 1, 3)

def concat_heads(heads):
    if not isinstance(heads, np.ndarray) or heads.ndim != 4:
        raise ValueError("heads must have shape (B,h,N,d_h)")
    b, h, n, d_h = heads.shape
    return heads.transpose(0, 2, 1, 3).reshape(b, n, h * d_h)

x = np.arange(48, dtype=np.float64).reshape(2, 3, 8)
heads = split_heads(x, 2)
recovered = concat_heads(heads)
EXPECTED_B0_N0_HEADS = np.array([[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0]], dtype=np.float64)
EXPECTED_B1_N2_HEADS = np.array([[40.0, 41.0, 42.0, 43.0], [44.0, 45.0, 46.0, 47.0]], dtype=np.float64)

### Answer check

In [ ]:
assert heads.shape == (2, 2, 3, 4)
assert recovered.shape == x.shape == (2, 3, 8)
assert heads.dtype == recovered.dtype == x.dtype == np.float64
np.testing.assert_allclose(heads[0, :, 0, :], EXPECTED_B0_N0_HEADS, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(heads[1, :, 2, :], EXPECTED_B1_N2_HEADS, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(recovered, x, atol=ATOL, rtol=RTOL)
assert heads[1, 1, 2, 3] == x[1, 2, 7]